# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imaniftikhar/week1_flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [5]:
import os
import getpass
import numpy as np
import pandas as pd
import duckdb
import lightgbm as lgb
from huggingface_hub import hf_hub_download

# 1. Download Parquet files from Hugging Face Warehouse
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste Hugging Face Token (hf_...): ')
token_str = HF_TOKEN.strip()
repo_id = "FlyRank/internship-warehouse"

fact_path = hf_hub_download(repo_id=repo_id, filename="fact_content_daily_performance_sample.parquet", repo_type="dataset", token=token_str)
dim_path = hf_hub_download(repo_id=repo_id, filename="dim_content.parquet", repo_type="dataset", token=token_str)

con = duckdb.connect()

# 2. Query Lane 2 Features
query = f"""
WITH aggregated_performance AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= CURRENT_DATE - INTERVAL '30 days' THEN gsc_clicks ELSE 0 END) AS clicks_last_30d,
        MAX(gsc_clicks) AS peak_clicks_30d
    FROM read_parquet('{fact_path}')
    GROUP BY content_hash_id
)
SELECT
    c.url_hash_id,
    c.client_hash_id,
    c.word_count,
    DATE_DIFF('day', c.content_updated_date::DATE, CURRENT_DATE) AS days_since_update,
    COALESCE(p.clicks_last_30d, 0) AS clicks_last_30d,
    COALESCE(p.peak_clicks_30d, 0) AS peak_clicks_30d
FROM read_parquet('{dim_path}') c
LEFT JOIN aggregated_performance p ON c.content_hash_id = p.content_hash_id
WHERE c.url_hash_id IS NOT NULL
"""

df = con.execute(query).df()

# 3. Preprocessing & Target Definition (Filter out minimal traffic noise)
df = df[df['peak_clicks_30d'] >= 10].copy()
df['word_count'] = df['word_count'].fillna(0)
df['log_peak_clicks'] = np.log1p(df['peak_clicks_30d'])
df['target_click_loss'] = (df['peak_clicks_30d'] - df['clicks_last_30d']).clip(lower=0)

features = ['days_since_update', 'log_peak_clicks', 'word_count']

# 4. Fit Model & Generate Scores
model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbosity=-1)
model.fit(df[features], df['target_click_loss'])
df['predicted_click_loss'] = model.predict(df[features])

# 5. Calibrated Action Rule Definitions (Fixes Lane 2 Refresh Routing)
def assign_action_and_reason(row):
    loss = row['predicted_click_loss']
    staleness = row['days_since_update']
    words = row['word_count']

    # Lowered staleness cutoff to 40 days to capture decaying high-loss content
    if loss > 15 and staleness > 40:
        return 'PRIORITY_REFRESH', 'STALE_CONTENT_HIGH_LOSS'
    elif loss > 15 and staleness <= 40:
        return 'TECHNICAL_AUDIT', 'RECENT_UPDATE_HIGH_LOSS'
    elif staleness > 365 and words < 400:
        return 'PRUNE_OR_CONSOLIDATE', 'LOW_TRAFFIC_THIN_CONTENT'
    else:
        return 'MONITOR', 'STABLE_PERFORMANCE'

df[['action', 'reason_code']] = df.apply(assign_action_and_reason, axis=1, result_type='expand')

# Rank queue strictly by predicted loss volume impact
action_queue = df.sort_values(by='predicted_click_loss', ascending=False)

print("=== Top 10 Priority Content Action Queue ===")
print(action_queue[['url_hash_id', 'days_since_update', 'peak_clicks_30d', 'predicted_click_loss', 'action', 'reason_code']].head(10).to_string(index=False))

Paste Hugging Face Token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Top 10 Priority Content Action Queue ===
         url_hash_id  days_since_update  peak_clicks_30d  predicted_click_loss           action             reason_code
url_464ff2db610e43bc                 58              890            199.287604 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_226c0d14bd4d70a5                 68              345            178.970115 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_4e03ecbb0ef86bd0                 49              228            176.434017 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_a120b548329280b6                 56              329            176.434017 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_f37479058155ca8e                 56               98            176.434017 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_b71a266de03ec98e                 54               68            176.434017 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_a246580e888e0aa3                 54               61            176.434017 PRIORITY_REFRESH STALE_CONTENT_HIGH_

**Queue Prioritization:** URLs are ranked strictly by estimated volume impact (predicted_click_loss), focusing editorial resources where potential traffic recovery is highest.

**Action Routing Logic:**
- PRIORITY_REFRESH (STALE_CONTENT_HIGH_LOSS): Content exceeding 40 days since last update with significant predicted traffic loss. Primary candidate pool for editorial refresh.

- TECHNICAL_AUDIT (RECENT_UPDATE_HIGH_LOSS): High predicted loss on content updated within the last 40 days, indicating potential technical SEO issues or search intent shifts.

- PRUNE_OR_CONSOLIDATE (LOW_TRAFFIC_THIN_CONTENT): Aged pages (>365 days) with under 400 words and low traffic volume.

- MONITOR (STABLE_PERFORMANCE): Stable or low-impact pages that require no immediate intervention.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Users:** Content strategists, SEO managers, and editorial teams responsible for prioritizing content maintenance workflows across client sites.

**Intended Scope:** Evaluates published, informational web pages with existing search engine history to identify candidates experiencing traffic decay due to staleness.

**Operational Limits:**
- Decision Support Only: Model outputs represent directional prioritization scores rather than guaranteed future traffic recovery or exact volume forecasts.

- New Domain Cold-Start: Requires at least 30 days of historical Google Search Console performance data; cannot accurately evaluate newly published domains or unindexed content.

- Non-Informational Pages: Excludes core transactional landing pages, product pages, and utility pages where traffic drops stem from inventory or conversion dynamics rather than content decay.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review Checklist (Mandatory prior to taking action):**

- SERP Feature & Intent Audit: Verify whether target search queries have undergone structural shifts (e.g., emergence of AI Overviews, direct answer boxes, or video carousels) before modifying body text.

- Brand & Compliance Verification: Ensure pages tagged for refresh do not contain active product discontinuation notices, legal disclosures, or regulated statements.

- Keyword Cannibalization Check: Confirm that proposed content updates do not conflict or overlap with higher-ranking active URLs on the same client domain.


**The No-Go List (Strictly Prohibited from Automation):**

- Automated Batch Rewriting: No direct LLM-based content overwriting or auto-publishing to production Content Management Systems (CMS) without human editorial review.

- Automated Deletions & Redirects: URLs tagged as PRUNE_OR_CONSOLIDATE must never be deleted, unpublished, or 301-redirected automatically.

- Core Revenue Pages: Top-converting landing pages and transactional hubs must be excluded from automated action queues.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Model Drift Trigger:** Retrain the model whenever Mean Absolute Error (MAE) on unseen client validation domains increases by $>15\%$ relative to baseline performance.
- **Search Engine Algorithm Trigger:** Initiate an immediate model retrain following confirmed Google Core Algorithm updates or major SERP layout changes.
- **Data Refresh Trigger:** Schedule monthly retraining cycles to incorporate newly ingested daily GSC performance data from the warehouse.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os

# 1. Ensure output directory exists
os.makedirs('work/outputs', exist_ok=True)

# 2. Export Action Queue to CSV
queue_export_path = 'work/outputs/content_action_queue.csv'
action_queue.to_csv(queue_export_path, index=False)

# 3. Generate and Export Summary Statistics
summary_df = action_queue['action'].value_counts().reset_index()
summary_df.columns = ['Action Category', 'URL Count']
summary_export_path = 'work/outputs/action_summary_stats.csv'
summary_df.to_csv(summary_export_path, index=False)

print(f"Action queue exported successfully to: {queue_export_path}")
print(f"Action summary stats exported successfully to: {summary_export_path}")
print("\n=== Action Queue Summary Statistics ===")
print(summary_df.to_string(index=False))

Action queue exported successfully to: work/outputs/content_action_queue.csv
Action summary stats exported successfully to: work/outputs/action_summary_stats.csv

=== Action Queue Summary Statistics ===
 Action Category  URL Count
         MONITOR        578
PRIORITY_REFRESH        354


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.